# 03. Baseline Model Training (U-Net)

This notebook trains a standard **U-Net** architecture natively on the 3m PlanetScope/Sentinel-2 dataset to establish a performance baseline for comparing against OlmoEarth and AlphaEarth foundation models.

In [ ]:
import os
import sys
sys.path.append('..')

import torch
from torch.utils.data import DataLoader, random_split
from src.data.dataset import RiverScopeDataset
from src.models.baseline import RiverScopeBaselineUNet
from src.training.trainer import RiverScopeTrainer

# 1. Setup Paths
DATA_ROOT = "../data/raw/RiverScope_dataset"
TRAIN_CSV = os.path.join(DATA_ROOT, "train.csv")

print("✅ Environment Ready.")

In [ ]:
# 2. Initialize Native Dataset
full_dataset = RiverScopeDataset(TRAIN_CSV, DATA_ROOT)

# Split into Train/Val (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], 
                                          generator=torch.Generator().manual_seed(42))

print(f"✅ Dataset Split: {len(train_dataset)} Train | {len(val_dataset)} Val")

In [ ]:
# 3. Initialize Baseline U-Net Model
# Note: The model is configured to handle the 12-channel input natively at full 3m resolution.
model = RiverScopeBaselineUNet(n_channels=12, n_classes=1)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"✅ U-Net Initialized on {device}")

In [ ]:
# 4. Define Trainers and Loaders
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

trainer = RiverScopeTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=1e-4
)

print("🚀 Starting Baseline Native Training Sweep...")
best_iou = trainer.fit(epochs=25, save_path='best_baseline_unet.pth')

print(f"\n✅ Baseline Sweep Complete! Max Native Validation IoU: {best_iou:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# 5. VISUAL COMPARISON
model.load_state_dict(torch.load('best_baseline_unet.pth', map_location=device))
model.eval()

vis_loader = DataLoader(val_dataset, batch_size=3, shuffle=True)
images, masks = next(iter(vis_loader))
images, masks = images.to(device), masks.to(device)

with torch.no_grad():
    logits = model(images)
    predictions = (torch.sigmoid(logits) > 0.5).float()

images_cpu = images.cpu()
masks_cpu = masks.cpu().squeeze()
preds_cpu = predictions.cpu().squeeze()

fig, axs = plt.subplots(3, 3, figsize=(15, 12))
for i in range(3):
    # RGBbands: B04 (Index 3), B03 (Index 2), B02 (Index 1)
    rgb = images_cpu[i, 0, [3, 2, 1], :, :].permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    
    axs[i, 0].imshow(rgb)
    axs[i, 0].set_title("Input RGB")
    
    axs[i, 1].imshow(masks_cpu[i], cmap='Blues')
    axs[i, 1].set_title("Ground Truth")
    
    axs[i, 2].imshow(preds_cpu[i], cmap='Blues')
    axs[i, 2].set_title("U-Net Baseline Prediction")

plt.tight_layout()
plt.show()